# Class 25: Scipy: Random Variable and Measurement Uncertainty
## Objective: Understand scipy tools appropriate to these numerical methods

These exercises are based on those in "Lecture 17: Scipy: Random Variable and Measurement Uncertainty" by Yuan-Sen Ting and available from https://tingyuansen.github.io/coding_essential_for_astronomers/lectures/lecture17-scipy-random-variable-measurement-uncertainty.html

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy import stats

# Plotting defaults
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

## Section 1: Random Variables and Estimators
In astronomy, we often treat a series of observations (like the measured brightness of a star over several nights) as a collection of **Random Variables**. Because our instruments, the Earth's atmosphere, and even the physics of light introduce noise, we use **statistical estimators** to determine the most likely "true" physical value.

* **Mean ($\mu$):** The arithmetic average. While common, it is highly sensitive to outliers (extreme "bad" data points).

$$\mu = \frac{1}{N} \sum_{i=1}^{N} x_i$$

* **Median:** The middle value of a sorted list. It is very robust; a single cosmic ray hit on a detector will move the mean significantly but will barely touch the median.
* **Dispersion:** Measures the "spread" or noise level of the data. This is typically expressed as **Variance** ($\sigma^2$) or **Standard Deviation** ($\sigma$).

$$\sigma = \sqrt{\frac{1}{N} \sum_{i=1}^{N} (x_i - \mu)^2}$$


In [ ]:
# Simulate 100 observations of a star with a "true" magnitude of 15.0
mean = 15
stdev = 0.2
nstars = 100
data = np.random.normal(loc=mean, scale=stdev, size=nstars)

mean_val = np.mean(data)
median_val = np.median(data)
std_val = np.std(data)

print(f"Mean: {mean_val:.3f}, Median: {median_val:.3f}, Std Dev: {std_val:.3f}")

plt.figure(figsize=(8,6))
plt.hist(data, range=(mean-5*stdev, mean+5*stdev), bins=20, density=True, alpha=0.6, color='g')

**Test your understanding:** 
Add a single "outlier" to the `data` array (e.g., a value of 100.0) and re-calculate the mean and median. Which one changed more?

In [ ]:
# Enter your code here 

## Section 2: The Normal (Gaussian) Distribution
The **Gaussian distribution** (or "Bell Curve") is the foundation of error analysis. According to the Central Limit Theorem, the sum of many independent random processes tends toward a Gaussian shape. This is why random measurement noise in telescopes usually follows this distribution.

It is defined entirely by two parameters: the mean ($\mu$) and the standard deviation ($\sigma$). The Probability Density Function (PDF) is:

$$f(x) = \frac{1}{\sigma\sqrt{2\pi}} \exp\left( -\frac{(x-\mu)^2}{2\sigma^2} \right)$$

In [ ]:
# Define a Gaussian with mu=0, sigma=1
mu, sigma = 0, 1
x = np.linspace(-5, 5, 200)
pdf = stats.norm.pdf(x, mu, sigma)

plt.plot(x, pdf, lw=2, label=f'Gaussian ($\mu$={mu}, $\sigma$={sigma})')
plt.fill_between(x, pdf, alpha=0.2)
plt.title("The Normal Distribution (PDF)")
plt.legend()
plt.show()

**Test your understanding:** Plot two Gaussians on the same graph: one with $\sigma=0.5$ and one with $\sigma=2.0$. How does the height and width of the curve change as the dispersion increases?

In [ ]:
# Enter your code here

## Section 3: Measurement Errors, Weighted Averages, and Outliers
In real research, some data points are "better" than others. Perhaps one night was cloudy (high error) and another was perfectly clear (low error). To get the best result, we use a **Weighted Average**, where each point $x_i$ is weighted by the inverse of its variance.

The weight $w_i$ is defined as:
$$w_i = \frac{1}{\sigma_i^2}$$

The weighted mean is then:
$$\bar{x}_{weighted} = \frac{\sum w_i x_i}{\sum w_i}$$

This naturally suppresses **outliers**—points that are far from the truth and have high uncertainties—preventing them from biasing our final result.

In [ ]:
measurements = np.array([10.1, 10.2, 12.5]) # 12.5 is a noisy outlier
errors = np.array([0.1, 0.1, 0.8])         # High error for the outlier

weights = 1 / errors**2
weighted_mean = np.sum(weights * measurements) / np.sum(weights)
simple_mean = np.mean(measurements)

print(f"Simple Mean: {simple_mean:.2f}")
print(f"Weighted Mean: {weighted_mean:.2f}")

**Test your understanding:** 
Why does the weighted mean stay closer to 10.15 than the simple mean? How does this help us handle "noisy" data points automatically?

## Section 4: Fitting Distributions
If we have a set of observations, we often want to "work backward" to find the parameters of the population. `scipy.stats` can take an array of data and estimate the best-fit $\mu$ and $\sigma$.

In [ ]:
# Generate noisy data
raw_data = stats.norm.rvs(loc=50, scale=5, size=500)

# Fit the data
mu_fit, sigma_fit = stats.norm.fit(raw_data)

plt.hist(raw_data, bins=30, density=True, alpha=0.6, color='g', label='Data Histogram')
x_range = np.linspace(30, 70, 100)
plt.plot(x_range, stats.norm.pdf(x_range, mu_fit, sigma_fit), 'r-', label='Best Fit Gaussian')
plt.legend()
plt.show()

**Test your understanding**
Change the `size` of `raw_data` from 500 to 10. How do the fitted parameters change? Does the fit look better or worse with fewer points?

In [ ]:
# Your code here

## Section 5: Cumulative Distributions and Confidence
The **Cumulative Distribution Function (CDF)** represents the probability that a random variable $X$ will be less than or equal to $x$. 

$$CDF(x) = P(X \le x) = \int_{-\infty}^{x} f(t) \, dt$$

In astronomy, we use "Sigma" ($\sigma$) to describe our confidence in a result. Because the area under the Gaussian PDF is known, we can translate $\sigma$ distances into probabilities:
* **$1\sigma$**: $\approx 68.3\%$ probability of being within this range.
* **$2\sigma$**: $\approx 95.4\%$ probability.
* **$3\sigma$**: $\approx 99.7\%$ probability.

In [ ]:
# Calculating the probability of a value falling within N-sigma
for n in [1, 2, 3, 5]:
    prob = stats.norm.cdf(n) - stats.norm.cdf(-n)
    print(f"{n}-sigma probability: {prob:.5f}")

**Test your understanding:** Astronomers often use "$5\sigma$" as the threshold for a new discovery (like the Higgs Boson or a new planet). Use stats.norm.cdf to calculate the probability of a data point being within $5\sigma$. How many "nines" of precision is that?

In [ ]:
# Your code here

## Solutions to In-Class Exercises

### Section 1 Solution

In [ ]:
# Create data with an outlier
data_with_outlier = np.append(data, 100.0)

new_mean = np.mean(data_with_outlier)
new_median = np.median(data_with_outlier)

print(f"Original Mean: {mean_val:.3f} -> New Mean: {new_mean:.3f}")
print(f"Original Median: {median_val:.3f} -> New Median: {new_median:.3f}")

# Solution: The Mean changes significantly (it is pulled toward the 100), 
# while the Median remains nearly the same

### Section 2 Solution

In [ ]:
x = np.linspace(-10, 10, 500)

plt.plot(x, stats.norm.pdf(x, 0, 0.5), label='Narrow ($\sigma=0.5$)')
plt.plot(x, stats.norm.pdf(x, 0, 2.0), label='Wide ($\sigma=2.0$)')
plt.title("Effect of Dispersion on Gaussian Shape")
plt.legend()
plt.show()

# Solution: As sigma increases, the distribution becomes shorter and wider. 
# Because the total area must always equal 1, a wider base requires a lower peak.

### Section 3 Solution

The weighted mean stays closer to the accurate values because the outlier (12.5) was assigned a very high uncertainty (0.8). Since weights are 1/sigma^2, the outlier's weight is much smaller than the others, effectively "ignoring" the bad data point in the final calculation.

### Section 4 Solution

In [ ]:
# Example Code
small_data = stats.norm.rvs(loc=50, scale=5, size=10)
mu_small, sigma_small = stats.norm.fit(small_data)

print(f"Fitted Mean (N=10): {mu_small:.2f}")

# Solution: With fewer points, the fitted mean and sigma will fluctuate 
# significantly away from the "true" values (50 and 5). The fit looks 
# much worse because the histogram is "clumpy" and doesn't represent 
# the underlying distribution well.

### Section 5 Solution

In [ ]:
# Calculate 5-sigma probability
prob_5sigma = stats.norm.cdf(5) - stats.norm.cdf(-5)

print(f"5-sigma probability: {prob_5sigma:.9f}")

# Solution: The probability is approximately 0.999999427. 
# This is "six nines" of precision. This incredibly high threshold 
# ensures that the result is almost certainly not a random fluke.